# М01. Лабораторная: три решения в одном исследовательском досье

**Авторская учебная адаптация · самостоятельная работа после [разбора](01_guided_case.ipynb).** Предпосылки: смысл logS, остатка, MAE/RMSE, графика пар и различие расчёта с наблюдением. [Словарь сокращений](GLOSSARY.md), [математический протокол](MATHEMATICAL_PROTOCOL.md), [установка](README.md#окружение).

## Аннотация и задача лабораторной

В разобранном исследовании средняя ошибка не смогла рассказать историю каждой записи. Теперь перед вами три продолжения той же работы. Сначала потребуется повторить знакомый анализ без готового решения. Затем — понять, почему уже проверенные данные после передачи говорят другое. Наконец — решить, отвечает ли доступное вычисление следующему химическому вопросу вообще.

**Зачем это делать:** в реальном исследовании редко достаточно запустить один метод. Нужно сохранить контрольный результат, локализовать несоответствие и выбрать наблюдение, которое устранит важную неопределённость. Научная опора здесь подлинная; организация передачи данных и запрос на следующее решение — учебная ситуация.

**Источники до начала:** [Delaney, 2004: статья](https://doi.org/10.1021/ci034243x), [авторская аннотация](https://pubmed.ncbi.nlm.nih.gov/15154768/); [точный CSV DeepChem](https://raw.githubusercontent.com/deepchem/deepchem/455d07f3e17e880a4d980b5b0365e4a02b417e14/datasets/delaney-processed.csv), [паспорт 1128 записей, единиц и SHA256](RESEARCH_SOURCE.md). Для третьего кейса понадобятся [экспериментальная структура 1HWK](https://www.rcsb.org/structure/1HWK) и [её первичная публикация](https://pubmed.ncbi.nlm.nih.gov/11349148/). Числа растворимости не становятся от этого данными по клеточной экспозиции.

## Что считается результатом

Вы сдаёте собственный код и наблюдаемые результаты, объяснение расхождения и исследовательское решение. Для каждого кейса сохраните короткую цепочку: **мой вопрос → что известно → почему выбран этот метод → что получилось → чего ещё не знаю → следующий шаг**. Файлы без этой связи не доказывают понимания.

Полные численные решения и преподавательский разбор находятся отдельно. Успешный запуск пустого шаблона не означает решение заданий. Приложение в M01 создавать не нужно; когда в следующих модулях потребуется CLI или интерфейс, работа продолжается в IDE — интегрированной среде разработки.

In [ ]:
from pathlib import Path
import sys, json, csv, math, statistics, hashlib, importlib.util, os
import matplotlib.pyplot as plt
from IPython.display import display, Image
MODULE = next(candidate for parent in [Path.cwd(), *Path.cwd().parents]
              for candidate in [parent, parent/'course_source/modules/M01']
              if (candidate/'m01.py').is_file())
if str(MODULE) not in sys.path:
    sys.path.insert(0, str(MODULE))
from m01 import META, MEASURED, PREDICTED, locate_snapshot, checksum, format_feedback, metric_invariants, create_dossier
OUT = Path(os.environ.get('M01_WORK_DIR', str(MODULE/'learner_work')))/'independent'
OUT.mkdir(parents=True, exist_ok=True)
print('Python:', sys.executable)
print('RDKit required: no; installed:', importlib.util.find_spec('rdkit') is not None)


## Входные условия: получить другие записи того же архива

Условия не меняются задним числом: raw-файл остаётся неизменным, шкала — log10(S/(mol/L)), правило выделения — `row_id % 4 == 1`. Эта часть из 282 строк не использовалась в guided. Она **не является независимой от обучения модели проверочной выборкой**: мы не знаем полный контекст происхождения готового столбца прогнозов.

Следующая ячейка только подготавливает вход и проверяет адреса строк. Решения она не вычисляет. `transfer` — список словарей с полями `row_id`, `name`, `smiles`, `measured`, `predicted`. Как обращаться к этим полям, показано в разборе.

In [ ]:
from m01 import read_rows,prepare_import_case
source_path=locate_snapshot();source_sha=checksum(source_path)
all_rows=read_rows(source_path)
transfer=[r for r in all_rows if r['row_id']%4==1]
guided_ids={r['row_id'] for r in all_rows if r['row_id']%4==0}
assert len(transfer)==282 and not guided_ids&{r['row_id'] for r in transfer}
display(transfer[:3])


## Кейс M01-C1. Другие записи: сохранится ли проблема среднего?

В разобранной части отдельная запись потребовала внимания, хотя среднее выглядело спокойнее. Для другой части архива такой вывод надо проверить заново. У вас нет основания ожидать ту же MAE или то же вещество с крупнейшим отклонением.

**Вопрос:** описывает ли средняя ошибка риск для каждого отдельного прогноза этой части? До кода запишите ожидание и наблюдение, которое заставит его пересмотреть. Укажите, что здесь является наблюдением, что — расчётом и почему пары сопоставимы по строкам.

**Математическая постановка:** примените определения остатка, MAE/RMSE и кратности из разбора к `transfer`. Сохраните знак, ID и все записи. Для описания локального риска используйте тот же объявленный учебный ориентир кратности; не выбирайте его после просмотра результата только ради красивого вывода. Если исследуете другой ориентир, покажите оба и объясните смену вопроса.

**Самостоятельное действие:** выберите и реализуйте сравнение, постройте график с подписями шкал, найдите крупнейшее расхождение и переведите его в понятную химическую величину. Сопоставьте результат с guided. Объясните разницу между «у этой части другая ошибка» и «исходная модель улучшилась».

**Артефакт:** CSV `row_id,name,measured,predicted,residual`, график, исходное ожидание и 3–5 предложений научного вывода. В `answer_1` верните `row_ids`, `MAE_logS`, `RMSE_logS`, `largest_row_id`, `conclusion`. Долю в пределах ориентира и его значение сохраните в собственном отчёте.

**Приёмка:** воспроизводится состав части, пары не потеряны; расчёт и знак согласованы; график позволяет проверить вывод; отдельное расхождение не заменено средним, а архивное сравнение не названо независимой проверкой модели. Формат словаря не гарантирует выполнение этих требований.

### Передача самому себе: чего не забыть для следующего кейса

Сохраните не только MAE, но и таблицу с ID, определение шкалы и SHA256. В следующем кейсе понадобится понять, изменилось ли вещество, метод или только представление числа. Без контрольного результата расследование превратится в догадку.

<details><summary>Подсказка 1 — постановка сравнения</summary>

Начните с пары значений одной строки в одной шкале. Спросите, что должно происходить со знаком остатка при завышенном прогнозе. Не требуется обучать модель заново.
</details>

<details><summary>Подсказка 2 — связь формулы и смысла</summary>

MAE усредняет модули, RMSE — квадраты с последующим корнем. Десятичный логарифм связывает разность logS с кратностью концентраций. Если получились отрицательная концентрация или «процент точности» без определения, вернитесь к величине, которую действительно считали.
</details>

In [ ]:
# Частично подготовлен вход; дальнейший расчёт и график напишите самостоятельно.
errors_c1=None
answer_1=None


### Что проверит следующая ячейка

Проверка формата обнаруживает отсутствующие поля; арифметические инварианты обнаруживают внутренне несогласованные числа. Она не сверяет ваш научный вывод с измерениями и не оценивает аргументацию. Пустое решение остаётся `NOT_RUN`, даже если сам notebook завершился без ошибки.

In [ ]:
print(format_feedback(answer_1,['row_ids','MAE_logS','RMSE_logS','largest_row_id','conclusion']))
if answer_1 is not None and errors_c1 is not None:
    print(metric_invariants(errors_c1,answer_1['MAE_logS'],answer_1['RMSE_logS']))


## Кейс M01-C2. После передачи результат изменился: где искать причину?

Вы сохранили контрольную таблицу. После её импорта получился другой результат. Предлагаются две поспешные реакции: удалить «плохие» вещества или немедленно заменить модель. Прежде чем выбирать одну из них, выясните, что именно изменилось между двумя вычислениями.

Эта ситуация передачи данных — **авторский учебный сценарий**. В производную копию намеренно внесена неисправность экспорта. Она не приписывается Delaney или DeepChem; исходные measured и raw CSV не меняются. Данные-основания взяты из того же подлинного snapshot.

**Вопрос:** несоответствие связано с составом записей, представлением величины, моделью или условиями наблюдения? Вход `import_case` содержит значения `reported_logS` и метаданные производителя. Название поля может конфликтовать с его определением. Не устанавливайте единицу по одному диапазону: положительная logS математически возможна.

**До исправления:** сформулируйте две гипотезы и назовите наблюдение, которое различит их. Сопоставьте ID и сохранённые контрольные величины C1. Затем проверьте значения вместе с метаданными. Исправление допустимо только после объяснения, что означает преобразуемая величина.

**Самостоятельное действие:** локализуйте расхождение, сохраните исходный импорт, выберите обоснованное преобразование, пересчитайте результат и сравните с C1 по тем же строкам. Проверьте неизменность исходника. Не изменяйте измерения ради совпадения метрики.

**Артефакт:** журнал «гипотеза → различающая проверка → наблюдение → вывод», исправленная производная CSV и единицы до/после. `answer_2`: `hypothesis`, `corrected_MAE_logS`, `raw_unchanged`, `rejected_hypothesis`, `evidence`.

**Приёмка:** объяснение опирается на смысл величины и метаданные; показана отвергнутая альтернатива; пары и исходник сохранены; восстановление доказано расчётом, а не ожидаемым ответом в тексте. В общем случае без метаданных допустим вывод «недостаточно данных для однозначного восстановления»; для нашей специально заданной неисправности необходимые метаданные предоставлены.

<details><summary>Одна подсказка — сначала определение величины</summary>

Сравните заголовок с `producer_quantity` и `producer_unit`, затем с контрольными парами C1. Найдите преобразование между установленными шкалами. Сам по себе более красивый график не доказывает правильность исправления.
</details>

In [ ]:
import_case=prepare_import_case(transfer)
display(import_case['metadata'])
display(import_case['rows'][:3])
# Собственная диагностика, исправление и проверка; готовой формулы здесь нет.
answer_2=None


In [ ]:
print(format_feedback(answer_2,['hypothesis','corrected_MAE_logS','raw_unchanged','rejected_hypothesis','evidence']))
assert checksum(source_path)==source_sha


### Что изменилось в вашем способе работы

В C1 вы отвечали на вопрос, что показывают числа. В C2 — почему одна процедура стала показывать другое. Оба раза нужная величина уже была в исходной постановке. Теперь придётся проверить саму постановку: то ли вообще измеряется, что нужно для следующего решения?

## Кейс M01-C3. В досье есть структура комплекса — хватит ли этого для вопроса о печени?

Вы начинаете собственное исследовательское досье B: аторвастатин и HMG-CoA-редуктаза, фермент, кодируемый геном **HMGCR**. В [PDB 1HWK](https://www.rcsb.org/structure/1HWK), записи банка трёхмерных структур Protein Data Bank, представлен экспериментальный комплекс аторвастатина с каталитической частью фермента. [Работа Istvan и Deisenhofer, 2001](https://pubmed.ncbi.nlm.nih.gov/11349148/) — первичный источник этого структурного контекста.

**Учебный запрос:** решить, можно ли доступными материалами обосновать достаточную внутриклеточную концентрацию вещества в выбранной клеточной системе печени для будущего опыта. Слово «достаточная» здесь пока не определено: вам предстоит связать его с конкретной целью, измеряемой величиной и критерием. Нужной концентрации, данных клеточного транспорта и измерений экспозиции пакет не содержит. Их нельзя придумать или взять из графика ESOL.

**Начальные условия:** карточки ниже, ваши результаты C1/C2, экспериментальная структура как источник сведений о комплексе. Условия будущего опыта, форма вещества и критерий достаточности должны быть явно определены в плане; они не являются найденными данными. Не переносите условия кристалла на клеточную систему без обоснования.

**Самостоятельная постановка:** сузьте общий запрос до опровержимого вопроса. Разделите опубликованный факт, вычисление и неизвестное. Выберите следующий метод и объясните, как его наблюдаемый результат уменьшит выбранную неопределённость. Сравните с одной разумной альтернативой. Структурный анализ, QSPR, биохимическая проверка действия на фермент и измерение клеточной экспозиции отвечают на разные вопросы — выбор зависит от вашей постановки.

**Математическая формализация плана:** назовите измеряемую переменную, единицу, сравниваемые условия и правило принятия решения. Например, для собственного выбранного endpoint можно записать символический порог и альтернативные исходы, но нельзя назначать необоснованное численное значение и выдавать его за научный норматив. Объясните, из какого источника или отдельного исследования потребуется получить порог. Укажите контроль, критерий опровержения и предел того, что подтвердит эксперимент.

**Артефакт:** `research_question.md`, запись источников и доказательств, `decisions.md`, новое объяснённое ребро `concept_map.json`. Связь карты должна иметь направление, смысл отношения, научное объяснение, `source_id`, ограничение и следующую проверку. Покажите, как контроль единиц и происхождения из C1/C2 войдёт в ваш новый план. Копия формулы из guided не заменит новую связь.

`answer_3`: `question`, `hypothesis`, `source_ids`, `method`, `control`, `falsification`, `limitation`, `next_action`. Алгоритмической подсказки здесь нет. Разные обоснованные вопросы могут вести к разным методам. Если исходное утверждение не подтверждено, явно обозначьте `insufficient_evidence` и объясните, какое именно доказательство отсутствует.

**Приёмка:** метод отвечает выбранному вопросу; переменные и условия определены; контроль и исход, опровергающий гипотезу, наблюдаемы; ограничение выводится из доступных свидетельств. Правильный научный ответ может заключаться в невозможности пока сделать исходный вывод. Он требует обоснования и плана, а не одной фразы «данных нет».

In [ ]:
cards=json.loads((MODULE/'data/evidence_cards.json').read_text(encoding='utf-8'))
display(cards)
dossier=create_dossier(OUT/'my_research')
print('Моё досье:',dossier)
answer_3=None


In [ ]:
print(format_feedback(answer_3,['question','hypothesis','source_ids','method','control','falsification','limitation','next_action']))
if answer_3 is not None:
    (dossier/'M01_submission.json').write_text(json.dumps(answer_3,ensure_ascii=False,indent=2),encoding='utf-8')


## Сдача, обсуждение и возврат к навыку

Каждый кейс оценивается по четырём направлениям: постановка и источник; метод и наблюдаемый артефакт; диагностика или контроль; решение с ограничением. По 0–1 баллу на направление, всего 12. Полбалла — релевантное, но неполное обоснование. Оценку научного содержания проводит преподаватель, рассматривая код, данные, результаты и объяснение вместе. Словарь с нужными ключами и успешный запуск не заменяют эту оценку.

Перед сдачей сделайте три коротких проверки понимания:

1. **Объяснение коллеге:** расскажите без кода, что пришлось решать в каждом кейсе и почему действие менялось.
2. **Контрпример:** назовите ситуацию, в которой ваш следующий выбранный метод не ответил бы на исходный вопрос.
3. **Самооценка по свидетельствам:** укажите одно решение, в котором уверены благодаря проверке, и одно, которое остаётся гипотезой. Назовите саму проверку.

На следующий день без подсказок восстановите смысл остатка и необходимый минимум журнала импорта. Затем проверьте себя. Выполнение этого возврата не фиксируется автоматически как доказанное обучение.

## Источники и помощь

- [Delaney (2004), DOI](https://doi.org/10.1021/ci034243x), [аннотация](https://pubmed.ncbi.nlm.nih.gov/15154768/); [фактический CSV](https://github.com/deepchem/deepchem/blob/455d07f3e17e880a4d980b5b0365e4a02b417e14/datasets/delaney-processed.csv), [происхождение и ограничения](RESEARCH_SOURCE.md).
- [PDB 1HWK](https://www.rcsb.org/structure/1HWK), [Istvan & Deisenhofer, 2001](https://pubmed.ncbi.nlm.nih.gov/11349148/) — экспериментальный структурный контекст C3.
- [Словарь](GLOSSARY.md), [математический протокол](MATHEMATICAL_PROTOCOL.md), [разбор](01_guided_case.ipynb). Преподавательские решения отделены от student notebook; публичный репозиторий не обеспечивает секретности ключей.